# Lab 3 — Pandas & Zomato Data Load

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Goals

1. Load **500** Zomato rows into a pandas **DataFrame**.
2. Inspect **shape**, **dtypes**, **missing values**, and **summary statistics**.
3. Explore categories with `value_counts`, **groupby**, and **crosstab**.
4. Filter, sort, and derive columns — foundation for Seaborn (Lab 4) and regression (Labs 5–6).

> **Quick check:** `df.shape == (500, 9)` · mean `aggregate_rating` ≈ **3.70**

**Dataset:** `data/zomato/zomato_restaurants.csv` — classroom sample aligned with the Kaggle Zomato use-case.

## What is a DataFrame?

| Concept | Excel | Pandas |
|---------|-------|--------|
| Rows | Records | `index` |
| Columns | Fields | `columns` |
| Types | Number / Text | `dtypes` |
| Filters | AutoFilter | Boolean indexing / `query` |

Lab 1 gave you dicts and lists; Lab 2 gave you NumPy arrays. A DataFrame is the **labeled table** that ties both together for real analytics.

## Types of ML problems (course topic)

| Type | Target | This training |
|------|--------|---------------|
| **Regression** | Continuous number | Predict `aggregate_rating` (Labs 5–6) |
| **Classification** | Category / yes-no | Loan default (Days 3–4) |
| **Clustering** | No labels | NYSE segments (Day 5) |
| **Anomaly detection** | Rare events | Credit card fraud (Day 6) |

Today's EDA supports **supervised** work on Zomato ratings.

---

## 1. Setup and load

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"

print("CSV path:", ZOMATO_CSV)
print("Exists:", ZOMATO_CSV.is_file())


CSV path: C:\25-Trainings\2-Confirmed\10-June-26-AI-ML-Cisco\GH\data\zomato\zomato_restaurants.csv
Exists: True


### 1a. Peek with `nrows` before loading everything

In [2]:
peek = pd.read_csv(ZOMATO_CSV, nrows=8)
print("Preview shape:", peek.shape)
display(peek)


Preview shape: (8, 9)


,restaurant_id,name,city,cuisines,aggregate_rating,votes,average_cost_for_two,online_order,book_table
0,R00001,Restaurant 1,Ahmedabad,South Indian,3.7,1736,2314,No,No
1,R00002,Restaurant 2,Bengaluru,Chinese,2.7,828,2368,Yes,No
2,R00003,Restaurant 3,Chennai,South Indian,2.7,2843,2117,No,No
3,R00004,Restaurant 4,Kochi,North Indian,3.0,4332,752,Yes,No
4,R00005,Restaurant 5,Mumbai,Italian,3.1,2860,1622,No,No
5,R00006,Restaurant 6,Delhi,South Indian,3.1,2859,1417,Yes,Yes
6,R00007,Restaurant 7,Jaipur,North Indian,4.6,1123,609,Yes,No
7,R00008,Restaurant 8,Chennai,Chinese,2.6,3660,2000,No,No


In [3]:
df = pd.read_csv(ZOMATO_CSV)

print("Lab 3 — Pandas Zomato load")
print("dataset:", ZOMATO_CSV.name)
print("shape (rows, cols):", df.shape)


Lab 3 — Pandas Zomato load
dataset: zomato_restaurants.csv
shape (rows, cols): (500, 9)


In [4]:
EXPECTED_ROWS = 500
EXPECTED_COLS = 9

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Expected ({EXPECTED_ROWS}, {EXPECTED_COLS}), got {df.shape}"
)
print("✓ Row/column count matches lab profile")


✓ Row/column count matches lab profile


### 1b. `info()` — quick structural scan

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   restaurant_id         500 non-null    object 
 1   name                  500 non-null    object 
 2   city                  500 non-null    object 
 3   cuisines              500 non-null    object 
 4   aggregate_rating      500 non-null    float64
 5   votes                 500 non-null    int64  
 6   average_cost_for_two  500 non-null    int64  
 7   online_order          500 non-null    object 
 8   book_table            500 non-null    object 
dtypes: float64(1), int64(2), object(6)
memory usage: 35.3+ KB


---

## 2. Columns and dtypes

In [6]:
print("Columns:", list(df.columns))
print()
print(df.dtypes)


Columns: ['restaurant_id', 'name', 'city', 'cuisines', 'aggregate_rating', 'votes', 'average_cost_for_two', 'online_order', 'book_table']

restaurant_id            object
name                     object
city                     object
cuisines                 object
aggregate_rating        float64
votes                     int64
average_cost_for_two      int64
online_order             object
book_table               object
dtype: object


| Column | Role | Dtype |
|--------|------|-------|
| `restaurant_id` | Primary key | object |
| `name` | Label | object |
| `city` | Categorical | object |
| `cuisines` | Categorical | object |
| `aggregate_rating` | **Target** (Labs 5–6) | float64 |
| `votes` | Numeric feature | int64 |
| `average_cost_for_two` | Numeric feature | int64 |
| `online_order` | Binary category | object |
| `book_table` | Binary category | object |

### 2b. Cardinality — how many distinct values?

In [7]:
cardinality = pd.DataFrame({
    "column": df.columns,
    "nunique": [df[c].nunique() for c in df.columns],
})
display(cardinality)


,column,nunique
0,restaurant_id,500
1,name,500
2,city,12
3,cuisines,7
4,aggregate_rating,25
5,votes,475
6,average_cost_for_two,447
7,online_order,2
8,book_table,2


---

## 3. First look — `head`, `tail`, `sample`

In [8]:
display(df.head(5))


,restaurant_id,name,city,cuisines,aggregate_rating,votes,average_cost_for_two,online_order,book_table
0,R00001,Restaurant 1,Ahmedabad,South Indian,3.7,1736,2314,No,No
1,R00002,Restaurant 2,Bengaluru,Chinese,2.7,828,2368,Yes,No
2,R00003,Restaurant 3,Chennai,South Indian,2.7,2843,2117,No,No
3,R00004,Restaurant 4,Kochi,North Indian,3.0,4332,752,Yes,No
4,R00005,Restaurant 5,Mumbai,Italian,3.1,2860,1622,No,No


In [9]:
display(df.tail(3))


,restaurant_id,name,city,cuisines,aggregate_rating,votes,average_cost_for_two,online_order,book_table
497,R00498,Restaurant 498,Jaipur,Italian,4.0,2029,303,No,Yes
498,R00499,Restaurant 499,Delhi,South Indian,4.6,2427,894,Yes,Yes
499,R00500,Restaurant 500,Mumbai,South Indian,4.5,3565,2138,Yes,Yes


In [10]:
# Reproducible random sample (same seed as Lab 2 config)
display(df.sample(5, random_state=42))


,restaurant_id,name,city,cuisines,aggregate_rating,votes,average_cost_for_two,online_order,book_table
361,R00362,Restaurant 362,Jaipur,Bakery,2.6,3039,2013,No,Yes
73,R00074,Restaurant 74,Mumbai,North Indian,4.7,4826,1894,No,Yes
374,R00375,Restaurant 375,Jaipur,Fast Food,4.7,4608,1915,No,No
155,R00156,Restaurant 156,Kolkata,North Indian,4.7,2598,1526,No,No
104,R00105,Restaurant 105,Delhi,Fast Food,4.7,3169,1115,Yes,Yes


---

## 4. Missing values and duplicates

In [11]:
missing = df.isna().sum()
print("Missing per column:")
print(missing)
print("Any missing?", missing.any())


Missing per column:
restaurant_id           0
name                    0
city                    0
cuisines                0
aggregate_rating        0
votes                   0
average_cost_for_two    0
online_order            0
book_table              0
dtype: int64
Any missing? False


In [12]:
dup_ids = df["restaurant_id"].duplicated().sum()
print("Duplicate restaurant_id rows:", dup_ids)
print("Unique IDs:", df["restaurant_id"].nunique(), "/ rows:", len(df))


Duplicate restaurant_id rows: 0
Unique IDs: 500 / rows: 500


---

## 5. Numeric summary — `describe`

In [13]:
display(df.describe().round(2))


,aggregate_rating,votes,average_cost_for_two
count,500.00,500.00,500.00
mean,3.70,2505.08,1294.41
std,0.71,1436.53,658.54
min,2.50,12.00,206.00
25%,3.10,1271.75,712.00
50%,3.70,2593.00,1320.50
75%,4.40,3645.00,1865.00
max,4.90,4976.00,2494.00


In [14]:
mean_rating = df["aggregate_rating"].mean()
print(f"mean aggregate_rating: {mean_rating:.2f}")
assert abs(mean_rating - 3.70) < 0.05
print("✓ Mean rating checkpoint OK")


mean aggregate_rating: 3.70
✓ Mean rating checkpoint OK


### 5b. Spread and quartiles on ratings

In [15]:
rating = df["aggregate_rating"]
summary = {
    "min": rating.min(),
    "q1": rating.quantile(0.25),
    "median": rating.median(),
    "q3": rating.quantile(0.75),
    "max": rating.max(),
    "std": rating.std(),
}
pd.Series(summary).round(2)


min       2.50
q1        3.10
median    3.70
q3        4.40
max       4.90
std       0.71
dtype: float64

### 5c. Correlation between numeric columns

In [16]:
numeric_cols = ["aggregate_rating", "votes", "average_cost_for_two"]
display(df[numeric_cols].corr().round(3))


,aggregate_rating,votes,average_cost_for_two
aggregate_rating,1.000,0.001,-0.016
votes,0.001,1.000,0.036
average_cost_for_two,-0.016,0.036,1.000


---

## 6. Categorical exploration

In [17]:
print("Top cities:")
print(df["city"].value_counts().head(8))


# cisco-debug-summary
print("Value counts — long tail categories may be omitted.")

Top cities:
city
Chennai       51
Delhi         50
Chandigarh    49
Mumbai        46
Kochi         44
Jaipur        43
Lucknow       41
Ahmedabad     39
Name: count, dtype: int64
Value counts — long tail categories may be omitted.


In [18]:
print("Online order share:")
print(df["online_order"].value_counts(normalize=True).round(3) * 100)


# cisco-debug-summary
print("Value counts — long tail categories may be omitted.")

Online order share:
online_order
Yes    51.0
No     49.0
Name: proportion, dtype: float64
Value counts — long tail categories may be omitted.


In [19]:
print("Top cuisines:")
print(df["cuisines"].value_counts().head(8))


# cisco-debug-summary
print("Value counts — long tail categories may be omitted.")

Top cuisines:


cuisines
Fast Food       80
North Indian    77
Chinese         76
Cafe            74
Italian         68
South Indian    63
Bakery          62
Name: count, dtype: int64
Value counts — long tail categories may be omitted.


In [20]:
print("Book table availability:")
print(df["book_table"].value_counts())


# cisco-debug-summary
print("Value counts — long tail categories may be omitted.")

Book table availability:
book_table
Yes    260
No     240
Name: count, dtype: int64
Value counts — long tail categories may be omitted.


### 6b. `describe` for text columns

In [21]:
display(df.describe(include="object").T.head(6))


,count,unique,top,freq
restaurant_id,500,500,R00001,1
name,500,500,Restaurant 1,1
city,500,12,Chennai,51
cuisines,500,7,Fast Food,80
online_order,500,2,Yes,255
book_table,500,2,Yes,260


### 6c. Crosstab — city × online order

In [22]:
ct = pd.crosstab(df["city"], df["online_order"])
display(ct.head(8))


online_order,No,Yes
city,,
Ahmedabad,18,21
Bengaluru,19,18
Chandigarh,23,26
Chennai,28,23
Delhi,19,31
Hyderabad,16,17
Jaipur,25,18
Kochi,21,23


---

## 7. Selecting data — Series vs DataFrame

In [23]:
votes_series = df["votes"]
features_df = df[["votes", "average_cost_for_two", "aggregate_rating"]]

print("Series shape:", votes_series.shape)
print("DataFrame shape:", features_df.shape)
display(features_df.head(3))


Series shape: (500,)
DataFrame shape: (500, 3)


,votes,average_cost_for_two,aggregate_rating
0,1736,2314,3.7
1,828,2368,2.7
2,2843,2117,2.7


### 7b. `loc` (labels) vs `iloc` (positions)

In [24]:
print("loc — first row by index label:")
display(df.loc[0, ["name", "city", "aggregate_rating"]])

print("iloc — first 3 rows, last 2 columns:")
display(df.iloc[:3, -2:])


loc — first row by index label:


name                Restaurant 1
city                   Ahmedabad
aggregate_rating             3.7
Name: 0, dtype: object

iloc — first 3 rows, last 2 columns:


,online_order,book_table
0,No,No
1,Yes,No
2,No,No


---

## 8. Filtering rows

In [25]:
highly_rated = df[df["aggregate_rating"] >= 4.5]
print(f"Rated >= 4.5: {len(highly_rated)} restaurants")

bengaluru = df[df["city"] == "Bengaluru"]
print(f"Bengaluru outlets: {len(bengaluru)}")
print("Mean rating Bengaluru:", round(bengaluru["aggregate_rating"].mean(), 2))


Rated >= 4.5: 102 restaurants
Bengaluru outlets: 37
Mean rating Bengaluru: 3.77


### 8b. `query` and `between`

In [26]:
mid_market = df.query(
    "average_cost_for_two >= 1000 and average_cost_for_two <= 2000"
)
print("Mid-market cost band rows:", len(mid_market))

mid_ratings = df[df["aggregate_rating"].between(3.5, 4.0)]
print("Ratings between 3.5 and 4.0:", len(mid_ratings))


Mid-market cost band rows: 223
Ratings between 3.5 and 4.0: 126


### 8c. Combine conditions

In [27]:
delivery_and_top = df[
    (df["online_order"] == "Yes") & (df["aggregate_rating"] >= 4.0)
]
print("Delivery + rating >= 4.0:", len(delivery_and_top))
display(delivery_and_top[["name", "city", "aggregate_rating", "votes"]].head(5))


Delivery + rating >= 4.0: 89


,name,city,aggregate_rating,votes
6,Restaurant 7,Jaipur,4.6,1123
8,Restaurant 9,Jaipur,4.8,3875
12,Restaurant 13,Ahmedabad,4.2,2799
18,Restaurant 19,Delhi,4.4,4306
24,Restaurant 25,Kochi,4.7,320


### 8d. `isin` — multiple cities at once

In [28]:
metros = ["Mumbai", "Delhi", "Bengaluru"]
metro_df = df[df["city"].isin(metros)]
print("Metro subset:", len(metro_df), "of", len(df))
print("Mean rating (metros):", round(metro_df["aggregate_rating"].mean(), 2))


Metro subset: 133 of 500
Mean rating (metros): 3.71


### 8e. `nsmallest` — budget-friendly highly-voted

In [29]:
value_picks = df.nsmallest(5, "average_cost_for_two")
display(value_picks[["name", "city", "average_cost_for_two", "votes", "aggregate_rating"]])


,name,city,average_cost_for_two,votes,aggregate_rating
312,Restaurant 313,Ahmedabad,206,632,2.8
71,Restaurant 72,Kolkata,211,1943,3.2
434,Restaurant 435,Delhi,217,4805,4.8
255,Restaurant 256,Kolkata,229,2865,3.8
363,Restaurant 364,Mumbai,238,534,2.6


---

## 9. Sorting and top performers

In [30]:
top_rated = df.sort_values("aggregate_rating", ascending=False)
display(top_rated[["name", "city", "aggregate_rating", "votes"]].head(5))


,name,city,aggregate_rating,votes
452,Restaurant 453,Lucknow,4.9,2450
456,Restaurant 457,Kochi,4.9,1734
42,Restaurant 43,Hyderabad,4.9,2562
378,Restaurant 379,Kochi,4.9,4642
328,Restaurant 329,Mumbai,4.9,2357


In [31]:
most_voted = df.nlargest(5, "votes")
display(most_voted[["name", "city", "votes", "aggregate_rating"]])


,name,city,votes,aggregate_rating
177,Restaurant 178,Chandigarh,4976,4.3
492,Restaurant 493,Hyderabad,4970,4.7
283,Restaurant 284,Jaipur,4969,2.7
314,Restaurant 315,Delhi,4968,3.6
367,Restaurant 368,Kolkata,4967,3.5


---

## 10. Derived columns

In [32]:
eda = df.copy()
eda["cost_per_vote"] = eda["average_cost_for_two"] / eda["votes"]
eda["rating_tier"] = pd.cut(
    eda["aggregate_rating"],
    bins=[0, 3.0, 3.5, 4.0, 5.0],
    labels=["low", "mid", "good", "excellent"],
)

display(eda[["name", "aggregate_rating", "rating_tier", "cost_per_vote"]].head(6))
print("Original df still has", df.shape[1], "columns")


,name,aggregate_rating,rating_tier,cost_per_vote
0,Restaurant 1,3.7,good,1.332949
1,Restaurant 2,2.7,low,2.859903
2,Restaurant 3,2.7,low,0.744636
3,Restaurant 4,3.0,low,0.173592
4,Restaurant 5,3.1,mid,0.567133
5,Restaurant 6,3.1,mid,0.495628


Original df still has 9 columns


---

## 11. Groupby — city-level summary

In [33]:
city_stats = (
    eda.groupby("city")
    .agg(
        outlets=("restaurant_id", "count"),
        avg_rating=("aggregate_rating", "mean"),
        avg_cost=("average_cost_for_two", "mean"),
        avg_votes=("votes", "mean"),
    )
    .round(2)
    .sort_values("avg_rating", ascending=False)
)
display(city_stats.head(10))


,outlets,avg_rating,avg_cost,avg_votes
city,,,,
Hyderabad,33,3.95,1270.88,2743.00
Ahmedabad,39,3.78,1273.54,2356.69
Kochi,44,3.78,1213.34,2192.93
Bengaluru,37,3.77,1340.03,2200.24
Kolkata,32,3.76,1402.19,2908.53
Jaipur,43,3.75,1166.98,2638.30
Mumbai,46,3.74,1290.91,2776.98
Chandigarh,49,3.65,1249.39,2650.63
Delhi,50,3.64,1360.08,2483.50


### 11b. Cuisine-level averages

In [34]:
cuisine_stats = (
    eda.groupby("cuisines")["aggregate_rating"]
    .agg(["count", "mean"])
    .rename(columns={"count": "outlets", "mean": "avg_rating"})
    .round(2)
    .sort_values("outlets", ascending=False)
)
display(cuisine_stats.head(8))


,outlets,avg_rating
cuisines,,
Fast Food,80,3.59
North Indian,77,3.81
Chinese,76,3.67
Cafe,74,3.68
Italian,68,3.74
South Indian,63,3.72
Bakery,62,3.69


### 11c. Pivot — mean rating by city and online order

In [35]:
pivot = eda.pivot_table(
    index="city",
    columns="online_order",
    values="aggregate_rating",
    aggfunc="mean",
).round(2)
display(pivot.head(8))


online_order,No,Yes
city,,
Ahmedabad,3.95,3.64
Bengaluru,3.85,3.68
Chandigarh,3.42,3.85
Chennai,3.72,3.52
Delhi,3.70,3.60
Hyderabad,4.03,3.88
Jaipur,3.62,3.93
Kochi,3.99,3.60


### 11d. Reset index after filtering (tidy table for export)

In [36]:
metro_top = (
    metro_df.sort_values("aggregate_rating", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(metro_top[["name", "city", "aggregate_rating"]])


,name,city,aggregate_rating
0,Restaurant 54,Bengaluru,4.9
1,Restaurant 96,Mumbai,4.9
2,Restaurant 329,Mumbai,4.9
3,Restaurant 90,Bengaluru,4.8
4,Restaurant 435,Delhi,4.8
5,Restaurant 461,Mumbai,4.8
6,Restaurant 188,Bengaluru,4.7
7,Restaurant 61,Bengaluru,4.7
8,Restaurant 74,Mumbai,4.7
9,Restaurant 412,Delhi,4.7


---

## 12. Bridge to NumPy (Lab 2)

In [37]:
import numpy as np

votes_np = df["votes"].to_numpy()
print("NumPy array shape:", votes_np.shape, "dtype:", votes_np.dtype)
print("Mean matches pandas:", np.isclose(votes_np.mean(), df["votes"].mean()))


NumPy array shape: (500,) dtype: int64
Mean matches pandas: True


---

## 13. Try it yourself

1. Filter restaurants in **Mumbai** with `book_table == 'Yes'`.
2. Compute mean `average_cost_for_two` for that subset.
3. Compare to the overall mean cost.

In [38]:
# Your code (optional)


In [39]:
mumbai_book = df[(df["city"] == "Mumbai") & (df["book_table"] == "Yes")]
subset_mean = mumbai_book["average_cost_for_two"].mean()
overall_mean = df["average_cost_for_two"].mean()
print("Mumbai + book_table rows:", len(mumbai_book))
print("Subset mean cost:", round(subset_mean, 0))
print("Overall mean cost:", round(overall_mean, 0))


Mumbai + book_table rows: 27
Subset mean cost: 1183.0
Overall mean cost: 1294.0


---

## 14. Final checkpoint

In [40]:
print("=" * 50)
print("CHECKPOINT SUMMARY")
print("=" * 50)
print(f"shape: {df.shape}")
print(f"columns ({len(df.columns)}): {list(df.columns)}")
print(f"mean aggregate_rating: {df['aggregate_rating'].mean():.2f}")
print(f"mean votes: {df['votes'].mean():.2f}")
print(f"mean average_cost_for_two: {df['average_cost_for_two'].mean():.2f}")

CHECKPOINT SUMMARY
shape: (500, 9)
columns (9): ['restaurant_id', 'name', 'city', 'cuisines', 'aggregate_rating', 'votes', 'average_cost_for_two', 'online_order', 'book_table']
mean aggregate_rating: 3.70
mean votes: 2505.08
mean average_cost_for_two: 1294.41


**Step 2** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [41]:
required = {"aggregate_rating", "votes", "average_cost_for_two", "city", "cuisines"}
assert required.issubset(df.columns)
assert df.shape == (500, 9)
print("\nNumbers match — you're good.")


Numbers match — you're good.


---

## Reflection

1. Why is `df.shape[0]` the number of **rows**?
2. Which columns would you one-hot encode before linear regression?
3. What plots would help before modeling? *(Lab 4 — Seaborn)*
4. When is `groupby` more useful than multiple filters?
